# RAG Notes Demo

**Projet :** IA Agents — Colab Files  
**Stack :** `google-genai` · `python-dotenv` · `numpy` · Python 3.12  
**Technique :** Retrieval-Augmented Generation (RAG) minimal sur fichiers `.txt` locaux

## Objectif

Ce notebook illustre une pipeline RAG minimale en trois étapes :

1. **Indexation** — lire des notes `.txt` depuis `assets/` et calculer leurs embeddings via Gemini.
2. **Recherche** — trouver les notes les plus proches d'une question par similarité cosine.
3. **Génération** — fournir le contexte retrouvé à Gemini pour produire une réponse précise et fondée.

> **Pré-requis :** Fichier `.env` à la racine du projet avec `GEMINI_API_KEY` renseigné.  
> Si `assets/` ne contient aucun `.txt`, le notebook crée `assets/example_note.txt` automatiquement.

In [ ]:
# ── Chargement de GEMINI_API_KEY depuis .env ──────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'à trouver requirements.txt."""
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError(
        "Racine du projet introuvable. "
        "Assurez-vous que requirements.txt existe à la racine."
    )


project_root = find_project_root(Path.cwd())
load_dotenv(dotenv_path=project_root / ".env")

api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY introuvable. "
        "Copiez .env.example en .env et renseignez votre clé."
    )

print(f"Racine du projet : {project_root}")
print("Clé API chargée avec succès.")

In [ ]:
# ── Lecture des fichiers .txt depuis assets/ ──────────────────────────────────
assets_dir = project_root / "assets"
txt_files = sorted(assets_dir.glob("*.txt"))

# Créer un exemple si aucun .txt n'est présent
if not txt_files:
    example_path = assets_dir / "example_note.txt"
    example_path.write_text(
        "Gemini est un modèle de langage multimodal développé par Google DeepMind.\n"
        "Il est accessible via l'API Gemini et le SDK officiel python google-genai.\n"
        "Le modèle gemini-2.0-flash est optimisé pour la rapidité et les tâches courantes.\n"
        "Pour générer du texte, on utilise client.models.generate_content().\n"
        "Les embeddings transforment du texte en vecteurs numériques de haute dimension.\n"
        "La similarité cosine mesure l'angle entre deux vecteurs : 1 = identiques, 0 = orthogonaux.",
        encoding="utf-8",
    )
    txt_files = [example_path]
    print(f"Aucun .txt trouvé — exemple créé : {example_path.name}")

# Charger le contenu de chaque fichier
notes = []
for path in txt_files:
    content = path.read_text(encoding="utf-8").strip()
    notes.append({"source": path.name, "content": content})
    print(f"  [OK] {path.name}  ({len(content)} caractères)")

print(f"\n{len(notes)} note(s) chargée(s).")

In [ ]:
# ── Création des embeddings avec Gemini ───────────────────────────────────────
from google import genai

EMBED_MODEL = "models/text-embedding-004"
GEN_MODEL   = "gemini-2.0-flash"

client = genai.Client(api_key=api_key)

print(f"Modèle d'embedding  : {EMBED_MODEL}")
print(f"Modèle de génération : {GEN_MODEL}")
print("Calcul des embeddings en cours...\n")

for note in notes:
    response = client.models.embed_content(
        model=EMBED_MODEL,
        contents=note["content"],
    )
    note["embedding"] = response.embeddings[0].values
    print(f"  [OK] {note['source']}  — dimension : {len(note['embedding'])}")

print(f"\nEmbeddings calculés pour {len(notes)} note(s).")

In [ ]:
# ── Fonction de similarité cosine ─────────────────────────────────────────────
import numpy as np


def cosine_similarity(vec_a: list, vec_b: list) -> float:
    a = np.array(vec_a, dtype=float)
    b = np.array(vec_b, dtype=float)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


# Vérifications rapides
assert cosine_similarity([1, 0], [1, 0]) == 1.0,  "Vecteurs identiques → 1"
assert cosine_similarity([1, 0], [0, 1]) == 0.0,  "Vecteurs orthogonaux → 0"
assert cosine_similarity([1, 0], [-1, 0]) == -1.0, "Vecteurs opposés → -1"

print("Fonction cosine_similarity opérationnelle.")

In [ ]:
# ── Recherche des notes les plus proches d'une question ───────────────────────

def retrieve(question: str, notes: list, top_k: int = 2) -> list:
    """Retourne les top_k notes classées par similarité décroissante."""
    q_response = client.models.embed_content(
        model=EMBED_MODEL,
        contents=question,
    )
    q_embedding = q_response.embeddings[0].values

    scored = [
        {
            "source":  note["source"],
            "content": note["content"],
            "score":   cosine_similarity(q_embedding, note["embedding"]),
        }
        for note in notes
    ]
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


# Test de recherche
question_test = "Comment utiliser Gemini pour générer du texte en Python ?"
resultats = retrieve(question_test, notes, top_k=2)

print(f"Question : {question_test}\n")
for i, r in enumerate(resultats, 1):
    print(f"  [{i}] {r['source']}  — score : {r['score']:.4f}")
    print(f"       {r['content'][:120]}...\n")

In [ ]:
# ── Génération d'une réponse à partir du contexte retrouvé ────────────────────

def answer(question: str, notes: list, top_k: int = 2) -> None:
    """Retrouve le contexte pertinent et génère une réponse avec Gemini."""
    retrieved = retrieve(question, notes, top_k=top_k)

    context = "\n\n".join(
        f"[Source : {r['source']}]\n{r['content']}"
        for r in retrieved
    )

    prompt = (
        "Tu es un assistant qui répond uniquement à partir du contexte fourni.\n"
        "Si la réponse n'est pas dans le contexte, dis-le explicitement.\n\n"
        f"Contexte :\n{context}\n\n"
        f"Question : {question}\n\n"
        "Réponds de façon concise et précise."
    )

    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt,
    )

    print(f"Question : {question}\n")
    print("=== Réponse Gemini ===")
    print(response.text)
    print("\n=== Sources utilisées ===")
    for r in retrieved:
        print(f"  - {r['source']}  (score : {r['score']:.4f})")


# Démonstration
answer("Quelle est la différence entre gemini-2.0-flash et gemini-2.5-pro ?")

## Prochaines améliorations

- [ ] **Persister les embeddings** : sauvegarder les vecteurs dans `outputs/` (JSON ou NumPy `.npy`) pour éviter de les recalculer à chaque session.
- [ ] **Découpage en chunks** : pour les notes longues, découper le texte en paragraphes avant l'indexation afin d'améliorer la précision de la recherche.
- [ ] **Multi-formats** : étendre le chargement à `.md` et `.pdf` en plus de `.txt`.
- [ ] **Interface interactive** : utiliser `ipywidgets` pour saisir la question directement dans le notebook sans modifier le code.
- [ ] **Base vectorielle** : remplacer la recherche linéaire par [ChromaDB](https://www.trychroma.com/) ou [FAISS](https://github.com/facebookresearch/faiss) pour monter en charge.
- [ ] **Évaluation** : mesurer la qualité des réponses avec un jeu de questions/réponses de référence.